# Standardizing Column Names and Data Formats

This notebook demonstrates how to clean and standardize column names and data formats in Pandas DataFrames to improve consistency, readability, and usability.

## Part 1: Loading and Inspecting Messy Data

We'll start by loading datasets with inconsistent column names and data formats.

In [ ]:
import pandas as pd
import numpy as np
import re
from datetime import datetime

# Load datasets with messy column names and formats
employees_df = pd.read_csv('../employees_messy_format.csv')
products_df = pd.read_csv('../products_messy_format.csv')

print("="*70)
print("ORIGINAL EMPLOYEES DATAFRAME (MESSY)")
print("="*70)
print(employees_df)

print("\n" + "="*70)
print("ORIGINAL PRODUCTS DATAFRAME (MESSY)")
print("="*70)
print(products_df)

### 1.1: Identifying Problems with Current Format

In [ ]:
print("PROBLEMS IN CURRENT DATA")
print("="*70)

print("\n1. INCONSISTENT COLUMN NAMES:")
print("-" * 70)
print("Current columns:", employees_df.columns.tolist())
print("\nIssues identified:")
print("  • 'Employee ID' - Contains space (should be underscore)")
print("  • 'emp_name' - Mixed case inconsistency")
print("  • 'DEPARTMENT' - All uppercase")
print("  • 'salary $' - Contains special character ($)")
print("  • 'hire DATE' - Inconsistent spacing/casing")
print("  • 'Phone #' - Contains special character (#)")

print("Impact:")
print("  → Hard to remember exact column names")
print("  → Easy to make typos when accessing columns")
print("  → Inconsistent in code (some use lowercase, some uppercase)")
print("  → Harder to merge with other datasets")

print("\n2. INCONSISTENT DATA FORMATS:")
print("-" * 70)
print("\nEmployee Names (Mixed Case):")
for name in employees_df['emp_name'].head(4):
    print(f"  {name}")
print("\n  Issue: Stored inconsistently (UPPERCASE, lowercase, Title Case)")
print("  Impact: Hard to deduplicate, search, or match names")

print("\nSalary Format (Currency String):")
for salary in employees_df['salary $'].head(3):
    print(f"  {salary}")
print("\n  Issue: Stored as text with $ and comma separators")
print("  Impact: Cannot perform numeric calculations")

print("\nHire Date Format (Multiple Formats):")
for date in employees_df['hire DATE'].head(5):
    print(f"  {date}")
print("\n  Issue: Different date formats (MM/DD/YYYY, YYYY-MM-DD, MM-DD-YYYY)")
print("  Impact: Hard to parse, compare, or sort dates")

print("\nPhone Number Format (Multiple Formats):")
for phone in employees_df['Phone #'].head(4):
    print(f"  {phone}")
print("\n  Issue: Inconsistent separators (- vs .)")
print("  Impact: Cannot standardize contact information")

## Part 2: Standardizing Column Names

Convert column names to a consistent format.

### 2.1: Basic Column Name Cleaning

Convert to lowercase, remove spaces, and replace with underscores.

In [ ]:
print("COLUMN NAME STANDARDIZATION - BASIC METHOD")
print("="*70)

print("\nOriginal column names:")
print(employees_df.columns.tolist())

# Method 1: Simple lowercase and replace spaces
employees_clean = employees_df.copy()
employees_clean.columns = employees_clean.columns.str.lower().str.replace(' ', '_')

print("\nAfter .str.lower().str.replace(' ', '_'):")
print(employees_clean.columns.tolist())

print("\nImprovement:")
print("  ✓ All lowercase (consistent)")
print("  ✓ Spaces replaced with underscores (valid Python identifiers)")
print("  ✗ Special characters still present (#, $)")

### 2.2: Advanced Column Name Cleaning

Remove special characters and standardize naming convention.

In [ ]:
print("COLUMN NAME STANDARDIZATION - ADVANCED METHOD")
print("="*70)

print("\nOriginal column names:")
print(employees_df.columns.tolist())

# Method 2: Complex cleaning
def clean_column_name(col):
    """
    Standardize column name:
    1. Convert to lowercase
    2. Replace spaces with underscores
    3. Remove special characters
    4. Remove multiple underscores
    """
    # Lowercase
    col = col.lower()
    # Replace spaces with underscore
    col = col.replace(' ', '_')
    # Remove special characters (keep only letters, numbers, underscores)
    col = re.sub(r'[^a-z0-9_]', '', col)
    # Remove multiple underscores
    col = re.sub(r'_+', '_', col)
    # Remove leading/trailing underscores
    col = col.strip('_')
    return col

employees_clean.columns = employees_clean.columns.map(clean_column_name)

print("\nAfter advanced cleaning:")
print(employees_clean.columns.tolist())

print("\nDetailed transformation:")
for orig, clean in zip(employees_df.columns, employees_clean.columns):
    print(f"  '{orig}' → '{clean}'")

print("\n✓ All improvements applied:")
print("  ✓ Lowercase (consistent casing)")
print("  ✓ Underscores replace spaces (valid identifiers)")
print("  ✓ Special characters removed (#, $)")
print("  ✓ Multiple underscores collapsed")

### 2.3: Using Built-in Methods for Cleaning

In [ ]:
print("USING PANDAS BUILT-IN METHODS FOR CLEANING")
print("="*70)

# Use .columns property with transformation
employees_v2 = employees_df.copy()

print("\nMethod 1: Using str accessor")
employees_v2.columns = (employees_v2.columns
                       .str.strip()  # Remove leading/trailing whitespace
                       .str.lower()  # Lowercase
                       .str.replace(' ', '_')  # Spaces to underscores
                       .str.replace(r'[^a-z0-9_]', '', regex=True))  # Remove special chars

print("Result:", employees_v2.columns.tolist())

print("\nMethod 2: Direct DataFrame.rename()")
ename_mapping = {col: col.lower().replace(' ', '_').replace('#', '').replace('$', '').strip('_')
                  for col in employees_df.columns}
employees_v3 = employees_df.rename(columns=rename_mapping)

print("Result:", employees_v3.columns.tolist())

### 2.4: Standardizing Products Column Names

In [ ]:
print("STANDARDIZING PRODUCTS COLUMN NAMES")
print("="*70)

print("\nOriginal product columns:")
print(products_df.columns.tolist())

# Apply the same cleaning function
products_clean = products_df.copy()
products_clean.columns = products_clean.columns.map(clean_column_name)

print("\nAfter standardization:")
print(products_clean.columns.tolist())

print("\nBefore and After:")
for orig, clean in zip(products_df.columns, products_clean.columns):
    print(f"  '{orig}' → '{clean}'")

## Part 3: Standardizing Data Formats

Apply consistent formatting to data columns.

### 3.1: Text Normalization - Employee Names

In [ ]:
print("TEXT NORMALIZATION - EMPLOYEE NAMES")
print("="*70)

print("\nOriginal name values:")
for i, name in enumerate(employees_clean['emp_name']):
    print(f"  {i}: '{name}'")

print("\nIssues:")
print("  • ALICE JOHNSON - All uppercase")
print("  • bob smith - All lowercase")
print("  • Charlie Brown - Title case (correct)")
print("  • diana WILLIAMS - Mixed case")
print("  • eve martinez - Lowercase")

print("\n" + "-"*70)
print("Apply Title Case Normalization")
print("-"*70)

employees_clean['emp_name'] = employees_clean['emp_name'].str.title()

print("\nAfter .str.title():")
for i, name in enumerate(employees_clean['emp_name']):
    print(f"  {i}: '{name}'")

print("\n✓ All names now in Title Case (consistent)")
print("✓ Easy to read and display")
print("✓ Suitable for proper names")

### 3.2: Numeric Formatting - Salary (Currency to Float)

### 3.3: Date Formatting - Standardizing to ISO Format

In [ ]:
print("DATE FORMATTING - HIRE DATE")
print("="*70)

print("\nOriginal date values (multiple formats):")
for i, date in enumerate(employees_clean['hire_date']):
    print(f"  {i}: '{date}'")

print("\nProblems:")
print("  • Mixed date formats (MM/DD/YYYY, YYYY-MM-DD, MM-DD-YYYY)")
print("  • Stored as strings")
print("  • Cannot compare or sort dates properly")
print("  • Ambiguous (is 02/03/04 Feb 3 or Mar 2?)")

print("\n" + "-"*70)
print("Convert to Datetime (ISO Format YYYY-MM-DD)")
print("-"*70)

# pd.to_datetime can infer many formats
employees_clean['hire_date'] = pd.to_datetime(employees_clean['hire_date'], infer_datetime_format=True)

print("\nAfter conversion:")
for i, date in enumerate(employees_clean['hire_date']):
    print(f"  {i}: {date} (type: {type(date).__name__})")

print("\n✓ All dates now in ISO format (YYYY-MM-DD)")
print("✓ Stored as datetime objects (not strings)")
print("✓ Can calculate tenure, sort dates, extract day/month/year")

print("\nExample extractions:")
print("  Days since hire date:")
today = pd.Timestamp('2024-04-15')
print(employees_clean[['emp_name', 'hire_date']].assign(
    days_employed=lambda x: (today - x['hire_date']).dt.days).head())

### 3.4: Categorical Standardization - Department

In [ ]:
print("CATEGORICAL STANDARDIZATION - DEPARTMENT")
print("="*70)

print("\nOriginal department values:")
print(employees_clean['department'].unique())

print("\nProblems:")
print("  • 'Engineering' vs 'engineering' (case inconsistency)")
print("  • 'SALES' vs 'Sales' (mixed case)")
print("  • Case-sensitive matching will see them as different values")

print("\n" + "-"*70)
print("Standardize to Title Case")
print("-"*70)

employees_clean['department'] = employees_clean['department'].str.title()

print("\nAfter standardization:")
print(employees_clean['department'].unique())

print("\n✓ All departments in consistent Title Case")
print("✓ Value counts now accurate:")
print(employees_clean['department'].value_counts())

### 3.5: Phone Number Standardization

In [ ]:
print("PHONE NUMBER STANDARDIZATION")
print("="*70)

print("\nOriginal phone numbers (mixed formats):")
for i, phone in enumerate(employees_clean['phone_']):
    print(f"  {i}: '{phone}'")

print("\nProblems:")
print("  • Inconsistent separators (- vs .)")
print("  • '555-0101' vs '555.0101' (should be standardized)")
print("  • Cannot reliably search or match")

print("\n" + "-"*70)
print("Standardize to XXX-XXXX Format")
print("-"*70)

def standardize_phone(phone):
    # Remove all non-digit characters
    digits = re.sub(r'\D', '', str(phone))
    # Format as XXX-XXXX (last 7 digits)
    if len(digits) >= 7:
        return f"{digits[-7:-4]}-{digits[-4:]}"
    return digits

employees_clean['phone'] = employees_clean['phone_'].apply(standardize_phone)
employees_clean = employees_clean.drop(columns=['phone_'])

print("\nAfter standardization:")
for i, phone in enumerate(employees_clean['phone']):
    print(f"  {i}: '{phone}'")

print("\n✓ All phone numbers use consistent XXX-XXXX format")
print("✓ No special characters, predictable format")
print("✓ Can search and match reliably")

### 3.6: Standardizing Products Data

In [ ]:
print("STANDARDIZING PRODUCTS DATA")
print("="*70)

print("\nOriginal product data:")
print(products_clean.head())

print("\n" + "-"*70)
print("Standardize product names")
print("-"*70)

print("\nBefore:")
print(products_clean['product_name'].tolist())

# Clean product names: remove special characters and parentheses, standardize case
products_clean['product_name'] = (products_clean['product_name']
                                   .str.replace(r'[()&]', '', regex=True)  # Remove parentheses and &
                                   .str.strip()  # Remove leading/trailing spaces
                                   .str.title())  # Title case

print("\nAfter:")
print(products_clean['product_name'].tolist())

print("\n" + "-"*70)
print("Standardize price to numeric")
print("-"*70)

print("\nBefore:")
print(products_clean['unit_price_'].head())

# Convert price to numeric
products_clean['unit_price'] = (products_clean['unit_price_']
                                 .str.replace('$', '')
                                 .str.replace(',', '')
                                 .astype(float))
products_clean = products_clean.drop(columns=['unit_price_'])

print("\nAfter:")
print(products_clean['unit_price'].tolist())

print("\n✓ Product names are now clean and consistent")
print("✓ Price values are now numeric (can calculate)")
print(f"  Average price: ${products_clean['unit_price'].mean():.2f}")

## Part 4: Before and After Comparison

In [ ]:
print("COMPLETE BEFORE AND AFTER COMPARISON")
print("="*70)

print("\n" + "="*70)
print("EMPLOYEES - BEFORE (MESSY)")
print("="*70)
print("\nColumn names:")
print(employees_df.columns.tolist())
print("\nData sample:")
print(employees_df.head(2))
print("\nData types:")
print(employees_df.dtypes)

print("\n" + "="*70)
print("EMPLOYEES - AFTER (STANDARDIZED)")
print("="*70)
print("\nColumn names:")
print(employees_clean.columns.tolist())
print("\nData sample:")
print(employees_clean.head(2))
print("\nData types:")
print(employees_clean.dtypes)

print("\n" + "-"*70)
print("IMPROVEMENTS SUMMARY")
print("-"*70)
print("\nColumn Names:")
for original, clean in zip(employees_df.columns, employees_clean.columns):
    if original != clean:
        print(f"  {original:20} → {clean}")

print("\nData Formatting:")
print("  emp_name: Mixed case → Title Case")
print("  salary: '$60,000' (string) → 60000.0 (float)")
print("  hire_date: Multiple formats (string) → 2019-06-20 (datetime)")
print("  department: Mixed case → Title Case")
print("  phone #: '555-0102' or '555.0102' → '555-0102' (consistent)")

print("\n✓ All column names follow snake_case convention")
print("✓ All column names are lowercase with underscores")
print("✓ Text fields are properly normalized")
print("✓ Numeric fields can be used in calculations")
print("✓ Dates can be compared and manipulated")
print("✓ Data is now consistent and usable")

## Part 5: Why Standardization Matters - The Merge Scenario

### 5.1: Merging Without Standardization (Problems)

In [ ]:
print("SCENARIO: MERGING TWO DATASETS")
print("="*70)

print("\nScenario: Merge employees with products they manage")
print("Dataset 1: Employees (original messy format)")
print("Dataset 2: Products (original messy format)")

print("\n" + "="*70)
print("ATTEMPT 1: Merge WITHOUT Standardization")
print("="*70)

# Create a sample products_employees relationship (messy format)
products_emp_messy = products_df.copy()
products_emp_messy['Employee_Name'] = ['ALICE JOHNSON', 'bob smith', 'charlie brown', 'Diana WILLIAMS', 'eve martinez', 'FRANK CHEN']  # Mixed case

print("\nTrying to merge on 'emp_name' from employees and 'Employee_Name' from products:")
print("Problem 1: Column names are different!")
print(f"  Employees has: 'emp_name'")
print(f"  Products has: 'Employee_Name'")

print("\n→ Must specify 'left_on' and 'right_on' parameters (error-prone)")

print("\nProblem 2: Even with correct column names, case doesn't match:")
print(f"  Employees: {employees_df['emp_name'].iloc[0]}")
print(f"  Products:  {products_emp_messy['Employee_Name'].iloc[0]}")
print("\n→ Merge fails because 'ALICE JOHNSON' != 'Alice Johnson'")

print("\nResult of attempted merge:")
merge_messy = pd.merge(employees_df, products_emp_messy,
                       left_on='emp_name',
                       right_on='Employee_Name',
                       how='left')
print(f"Expected rows: 6 (should match all employees with products)")
print(f"Actual rows: {len(merge_messy)} (many don't match due to case differences)")
print("\n✗ Merge produced unexpected results due to:")
print("  • Inconsistent column naming")
print("  • Inconsistent data formatting (case)")

### 5.2: Merging WITH Standardization (Solution)

In [ ]:
print("\n" + "="*70)
print("ATTEMPT 2: Merge WITH Standardization")
print("="*70)

# Prepare products with standardized employee names
products_emp_clean = products_clean.copy()
products_emp_clean['emp_name'] = ['Alice Johnson', 'Bob Smith', 'Charlie Brown', 'Diana Williams', 'Eve Martinez', 'Frank Chen']
products_emp_clean['emp_name'] = products_emp_clean['emp_name'].str.title()  # Standardize

print("\nAfter standardization:")
print(f"  Employees column: 'emp_name' (consistent naming)")
print(f"  Products column: 'emp_name' (matching name & format)")

print("\nEmployee names:")
for name in employees_clean['emp_name'].head(3):
    print(f"  {name}")

print("\nProduct employee names:")
for name in products_emp_clean['emp_name'].head(3):
    print(f"  {name}")

print("\n→ Simple join on 'emp_name' (same column name, consistent format):")
merge_clean = pd.merge(employees_clean[['emp_name', 'department', 'salary']],
                       products_emp_clean[['product_name', 'unit_price', 'emp_name']],
                       on='emp_name',
                       how='inner')

print(f"Expected rows: 6 (should match all employees with products)")
print(f"Actual rows: {len(merge_clean)} ✓ Perfect match!")

print("\nMerged data:")
print(merge_clean[['emp_name', 'product_name', 'unit_price', 'salary']])

print("\n✓ Benefits of standardization:")
print("  • Same column name ('emp_name') in both datasets")
print("  • Consistent data format (Title Case)")
print("  • Predictable merge results (all rows matched)")
print("  • Fewer errors and easier to understand code")
print("  • Maintainable and scalable")

## Part 6: Implementation Checklist

In [ ]:
print("DATA STANDARDIZATION CHECKLIST")
print("="*70)

print("\n☐ STEP 1: ANALYZE CURRENT STATE")
print("  ☐ Load data and inspect column names")
print("  ☐ Check data types (str, float, datetime, etc.)")
print("  ☐ Identify inconsistencies in column naming")
print("  ☐ Identify inconsistencies in data formatting")
print("  ☐ Document problems (create standardization plan)")

print("\n☐ STEP 2: STANDARDIZE COLUMN NAMES")
print("  ☐ Convert all to lowercase")
print("  ☐ Replace spaces with underscores")
print("  ☐ Remove special characters (#, $, %, etc.)")
print("  ☐ Use snake_case convention throughout")
print("  ☐ Make names descriptive but concise")

print("\n☐ STEP 3: STANDARDIZE TEXT DATA")
print("  ☐ Identify inconsistent casing")
print("  ☐ Apply consistent casing (Title, lowercase, UPPERCASE)")
print("  ☐ Strip leading/trailing whitespace")
print("  ☐ Remove unnecessary special characters")
print("  ☐ Verify no duplicate variations exist")

print("\n☐ STEP 4: STANDARDIZE NUMERIC DATA")
print("  ☐ Remove currency symbols ($, €, etc.)")
print("  ☐ Remove thousand separators (,)")
print("  ☐ Convert strings to proper numeric types (int, float)")
print("  ☐ Handle negative numbers properly")
print("  ☐ Test that calculations work correctly")

print("\n☐ STEP 5: STANDARDIZE DATE DATA")
print("  ☐ Identify current date format(s)")
print("  ☐ Convert all to datetime objects")
print("  ☐ Use ISO format (YYYY-MM-DD) for consistency")
print("  ☐ Verify dates are sensible (no future hire dates, etc.)")
print("  ☐ Test date calculations work")

print("\n☐ STEP 6: VERIFY & DOCUMENT")
print("  ☐ Check all column names follow naming convention")
print("  ☐ Verify correct data types for all columns")
print("  ☐ Spot-check sample data for correctness")
print("  ☐ Add comments explaining standardization choices")
print("  ☐ Document any removed/modified values")

print("\n" + "="*70)
print("EXAMPLE CODE STRUCTURE")
print("="*70)

print("\n# 1. Load data")
print("df = pd.read_csv('data.csv')")

print("\n# 2. Standardize column names")
print("df.columns = (df.columns")
print("    .str.lower()")
print("    .str.replace(' ', '_')")
print("    .str.replace(r'[^a-z0-9_]', '', regex=True))")

print("\n# 3. Standardize text")
print("df['name'] = df['name'].str.title()")

print("\n# 4. Standardize numeric")
print("df['price'] = df['price'].str.replace('$', '').astype(float)")

print("\n# 5. Standardize dates")
print("df['date'] = pd.to_datetime(df['date'])")

print("\n# 6. Verify")
print("print(df.dtypes)")
print("print(df.head())")

## Summary: Key Standardization Techniques

### Column Name Standardization
- **`.lower()`**: Convert to lowercase
- **`.replace(' ', '_')`**: Replace spaces with underscores
- **`.str.replace(r'[^a-z0-9_]', '', regex=True)`**: Remove special characters
- **Snake case convention**: lowercase_with_underscores

### Text Standardization
- **`.title()`**: Convert to Title Case (for names)
- **`.lower()`**: Convert to lowercase (for categories)
- **`.upper()`**: Convert to UPPERCASE (less common)
- **`.strip()`**: Remove leading/trailing whitespace

### Numeric Standardization
- **Remove currency**: `.str.replace('$', '')`
- **Remove separators**: `.str.replace(',', '')`
- **Convert type**: `.astype(float)` or `.astype(int)`

### Date Standardization
- **Parse dates**: `pd.to_datetime(column, infer_datetime_format=True)`
- **Format dates**: `.dt.strftime('%Y-%m-%d')`
- **Extract parts**: `.dt.year`, `.dt.month`, `.dt.day`

### Why Standardization Matters
- **Merging/Joining**: Consistent column names prevent join failures
- **Calculations**: Proper data types enable arithmetic and comparisons
- **Searching**: Standardized text makes matching predictable
- **Readability**: Snake_case is more Pythonic and readable
- **Maintainability**: Consistent conventions reduce errors
- **Reproducibility**: Others can understand and work with your data